# Agent 2 · Trajectoire

Le réseau choisit sept points dans le corridor visible et une vitesse indicative. Le rendu utilise le même plan que le pilote optionnel.

In [ ]:
from dataclasses import dataclass
import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

# Coordonnées image normalisées, origine en haut à gauche. De près vers loin.
LIGNES = np.array([.87,.82,.77,.72,.67,.62,.57], np.float32)
N = len(LIGNES)
# Vue cockpit Abarth 500 (320 × 240). Bande de bitume au-dessus du volant,
# entre les montants. Les sorties restent dans le repère de l'image entière.
LIGNES_COCKPIT = np.linspace(116/239, 109/239, N, dtype=np.float32)
COCKPIT_ABARTH = 'cockpit_abarth500_v1'
# Vue capot Abarth 500 (320 × 240). Bande de bitume au-dessus du capot et du
# bandeau Essentials (y >= 182), jusqu'aux abords de l'horizon pour anticiper les virages.
LIGNES_CAPOT = np.linspace(176/239, 126/239, N, dtype=np.float32)
CAPOT_ABARTH = 'capot_abarth500_v1'
CADRAGES = {'historique': LIGNES, COCKPIT_ABARTH: LIGNES_COCKPIT, CAPOT_ABARTH: LIGNES_CAPOT}

class GuidageIndisponible(ValueError):
    pass

@dataclass(frozen=True)
class Trajectoire:
    xy: np.ndarray           # N couples (x,y) dans [0,1], près -> loin
    cible_kmh: float

def couloir(probabilite, seuil=.65, cadrage="historique", retour_lignes=False, recul_haut=0):
    p = np.asarray(probabilite)
    if p.ndim != 2 or min(p.shape)<32 or not np.isfinite(p).all():
        raise GuidageIndisponible("Carte de piste invalide")
    if cadrage not in CADRAGES:
        raise ValueError(f"Cadrage inconnu : {cadrage}")
    if cadrage != 'historique':
        # La carte du U-Net peut faire 128 × 96 ou 256 × 144. Conserver ses décisions
        # par voisin le plus proche, puis mesurer la marge en pixels caméra.
        p = cv2.resize(p.astype(np.float32), (320,240), interpolation=cv2.INTER_NEAREST)
    brut = (p >= seuil).astype(np.uint8)
    if cadrage == CAPOT_ABARTH:
        # Combler uniquement les micro-trous d'ombre strictement enclos (probabilité moyenne >= 0.30),
        # jamais un obstacle (p ~ 0) ni une coupure reliée au bord de piste.
        inv = (1 - brut).copy()
        ff = np.zeros((inv.shape[0]+2, inv.shape[1]+2), np.uint8)
        for sx, sy in ((0,0),(inv.shape[1]-1,0),(0,inv.shape[0]-1),(inv.shape[1]-1,inv.shape[0]-1)):
            if inv[sy, sx] == 1:
                cv2.floodFill(inv, ff, (sx, sy), 2)
        trous = (inv == 1).astype(np.uint8)
        if trous.any():
            nb_t, lab_t, st_t, _ = cv2.connectedComponentsWithStats(trous)
            for k in range(1, nb_t):
                if int(st_t[k, cv2.CC_STAT_AREA]) <= 400 and float(p[lab_t == k].mean()) >= .30:
                    brut[lab_t == k] = 1
    # Erosion avant le choix des points pour garder une marge aux obstacles.
    libre = cv2.erode(brut, np.ones((3,3),np.uint8))
    h,w = libre.shape
    if libre.mean() > .92:
        raise GuidageIndisponible("Aucune limite de piste identifiable")
    if cadrage == COCKPIT_ABARTH:
        libre[:,:118] = 0
        libre[:,298:] = 0
    if cadrage == CAPOT_ABARTH:
        # Portée adaptative jusqu'à l'horizon visible pour anticiper les virages tôt
        # avec échantillonnage perspectif concentré sur la zone d'entrée de virage.
        y_pres, y_min = 176, 108
        min_l = max(12, int(.055*w))
        par_y, precedent, y_loin = {}, None, y_pres
        for y in range(y_pres, y_min-1, -1):
            transitions = np.diff(np.r_[0, libre[y], 0].astype(int))
            segments = [(a/(w-1),(b-1)/(w-1)) for a,b in
                        zip(np.where(transitions==1)[0],np.where(transitions==-1)[0])
                        if b-a >= min_l]
            if precedent is None:
                segments = [(a,b) for a,b in segments if a <= .5 <= b]
                reference = .5
            else:
                segments = [(a,b) for a,b in segments if min(b,precedent[1]) > max(a,precedent[0])]
                reference = sum(precedent)/2
            if not segments:
                break
            precedent = min(segments,key=lambda ab: abs(sum(ab)/2-reference))
            par_y[y] = precedent
            y_loin = y
        y_top = max(117, y_loin + 3) + int(recul_haut)
        if y_pres - y_top < 24:
            raise GuidageIndisponible("Aucun passage continu devant la voiture")
        u = np.linspace(0, 1, N, dtype=np.float32) ** .62
        ys = np.rint(y_pres - (y_pres - y_top)*u).astype(int)
        bornes = np.asarray([par_y[int(y)] for y in ys], np.float32)
        lignes = (ys / (h-1)).astype(np.float32)
        return (bornes, libre, lignes) if retour_lignes else (bornes, libre)
    lignes = CADRAGES[cadrage]
    precedent = None
    limites = []
    for y in np.rint(lignes*(h-1)).astype(int):
        transitions = np.diff(np.r_[0, libre[y], 0].astype(int))
        segments = [(a/(w-1),(b-1)/(w-1)) for a,b in
                    zip(np.where(transitions==1)[0],np.where(transitions==-1)[0])
                    if b-a >= max(4,.035*w)]
        if precedent is None:
            segments = [(a,b) for a,b in segments if a <= .5 <= b]
            reference = .5
        else:
            segments = [(a,b) for a,b in segments if min(b,precedent[1]) > max(a,precedent[0])]
            reference = sum(precedent)/2
        if not segments:
            raise GuidageIndisponible("Aucun passage continu devant la voiture")
        precedent = min(segments,key=lambda ab: abs(sum(ab)/2-reference))
        limites.append(precedent)
    bornes = np.asarray(limites,np.float32)
    return (bornes, libre, lignes) if retour_lignes else (bornes, libre)

class Planificateur(nn.Module):
    def __init__(self):
        super().__init__()
        self.cadrage = 'historique'
        self.reseau = nn.Sequential(nn.Linear(2*N+2,64),nn.Tanh(),nn.Linear(64,64),
                                    nn.Tanh(),nn.Linear(64,N+1))

    def forward(self, bornes, etat):
        sortie = self.reseau(torch.cat((bornes.flatten(1),etat),dim=1))
        bas,haut = bornes[:,:,0],bornes[:,:,1]
        x = bas + (haut-bas)*(.1 + .8*sortie[:,:N].sigmoid())
        vitesse = 12 + 38*sortie[:,N].sigmoid()
        return x,vitesse

    @torch.inference_mode()
    def proposer(self, probabilite, vitesse_ms, lacet_rads):
        etat = np.asarray([vitesse_ms/30,lacet_rads/3],np.float32)
        if not np.isfinite(etat).all() or vitesse_ms<0:
            raise GuidageIndisponible("Télémétrie invalide")
        reculs = (0, 3, 6) if self.cadrage == CAPOT_ABARTH else (0,)
        derniere = None
        for recul in reculs:
            try:
                bornes, libre, lignes = couloir(probabilite, cadrage=self.cadrage, retour_lignes=True, recul_haut=recul)
            except GuidageIndisponible as exc:
                if recul == 0:
                    raise
                derniere = exc
                break
            x,v = self(torch.from_numpy(bornes)[None],torch.from_numpy(etat)[None])
            x0 = x[0].numpy()
            mid = bornes.mean(axis=1)
            h,w = libre.shape
            if self.cadrage == CAPOT_ABARTH:
                u_fit = np.linspace(0., 1., N, dtype=np.float32) ** .62
                w_fit = np.array([1.8, 1.0, 1.0, 1.0, 1.0, 1.3, 1.5], np.float32)
                larg = np.maximum(bornes[:,1] - bornes[:,0], .04)
                lo_c, hi_c = bornes[:,0] + .12*larg, bornes[:,1] - .12*larg
                x_arc = np.clip(np.polyval(np.polyfit(u_fit, x0, 2, w=w_fit), u_fit).astype(np.float32), lo_c, hi_c)
                m_arc = np.clip(np.polyval(np.polyfit(u_fit, mid, 2, w=w_fit), u_fit).astype(np.float32), lo_c, hi_c)
                candidats = (x_arc, .75*x_arc + .25*m_arc, .45*x_arc + .55*m_arc, m_arc, x0, mid)
            else:
                candidats = tuple((1.-r)*x0 + r*mid for r in (0., .35, .70, 1.0))
            for xc in candidats:
                points = np.column_stack((xc, lignes))
                ligne = np.zeros_like(libre)
                pixels = np.rint(points*[w-1,h-1]).astype(np.int32)
                cv2.polylines(ligne,[pixels],False,1,2)
                if not np.any((ligne==1) & (libre==0)):
                    return Trajectoire(points,float(v.item()))
        raise derniere or GuidageIndisponible("Trajectoire interrompue par une bordure ou un obstacle")

def entrainer_planificateur(iterations=2600, seed=73):
    # Optimisation trajectoire de course (ancrée devant le capot -> corde intérieure -> sortie)
    # par projection perspective 3D (lignes droites, approches de virages, épingles,
    # chicanes) + robustesse aux morsures d'ombre + corridors classiques.
    torch.manual_seed(seed)
    modele = Planificateur()
    optim = torch.optim.Adam(modele.parameters(),lr=.003)
    t_lin = torch.linspace(0,1,N)[None]
    u_cap = torch.linspace(0,1,N)[None] ** .62
    profil_s = torch.tensor([-0.06, 0.04, 0.14, 0.22, 0.24, 0.20, 0.12])[None]
    for it in range(iterations):
        if it in (iterations//2, 3*iterations//4):
            for g in optim.param_groups:
                g['lr'] *= .5
        B, B_3d = 128, 96
        B_std = B - B_3d
        r = torch.rand(B_3d, 12)
        z0 = 4.6 + 1.4*r[:,0:1]
        alpha = .80 + .10*r[:,1:2]
        Z = z0 / (1.0 - alpha*u_cap)
        W = 9.5 + 5.5*r[:,2:3]
        foc = .62 + .36*r[:,3:4]
        X0 = (r[:,4:5] - .5)*.78*W
        psi = (r[:,5:6] - .5)*.30
        rt = r[:,6:7]
        is_straight = (rt < .26).float()
        is_chicane = (rt >= .84).float()
        is_single = 1.0 - is_straight - is_chicane
        sign_k = torch.where(r[:,7:8] < .5, -1.0, 1.0)
        mag_k = .005 + .046*(r[:,8:9]**1.15)
        kappa = sign_k*mag_k*(1.0 - is_straight)
        z_start = torch.where(r[:,9:10] < .42, torch.zeros_like(r[:,9:10]), 3.0 + 15.0*r[:,10:11])
        dz = F.relu(Z - z_start)
        z_mid = z_start + 13.0 + 8.0*r[:,11:12]
        dz2 = F.relu(Z - z_mid)
        Xc = -X0 - psi*Z + .5*kappa*(dz**2) - is_chicane*.85*kappa*(dz2**2)
        bas_3d = (.5 + foc*(Xc - .5*W)/Z).clamp(0., .94)
        haut_3d = torch.maximum(bas_3d+.06, (.5 + foc*(Xc + .5*W)/Z).clamp(.06, 1.))
        s_car = (.5 + X0/W).clamp(.16, .84)
        intens = torch.tanh(mag_k/.0065)*(1.0 - is_straight)
        s_apex_raw = (.5 + .35*sign_k*intens).clamp(.15, .85)
        s_apex = torch.where(sign_k < 0, torch.minimum(s_car, s_apex_raw), torch.maximum(s_car, s_apex_raw))
        ramp_turn = torch.sigmoid((Z - (z_start - 5.0))/4.5)
        s_single = (1.0 - ramp_turn)*s_car + ramp_turn*s_apex
        ramp_str = ((Z - 4.5)/24.0).clamp(0., 1.)
        s_str = (1.0 - ramp_str)*s_car + ramp_str*.5
        s_apex2 = (.5 - .32*sign_k*intens).clamp(.16, .84)
        ramp2 = torch.sigmoid((Z - (z_mid - 2.0))/4.0)
        s_chic = (1.0 - ramp2)*s_single + ramp2*s_apex2
        s_opt = (is_straight*s_str + is_single*s_single + is_chicane*s_chic).clamp(.15, .85)
        X_opt = Xc + (s_opt - .5)*W
        w_merge = ((Z - 4.3)/9.5).clamp(0., 1.) ** 1.15
        x_proj = .5 + foc*(w_merge*X_opt)/Z
        x_far = x_proj[:,-1:]
        x_mono = x_proj.clamp(min=torch.minimum(x_proj[:,:1], x_far), max=torch.maximum(x_proj[:,:1], x_far))
        x_proj = is_single*x_mono + (1.0 - is_single)*x_proj
        larg_3d = (haut_3d - bas_3d).clamp(min=.04)
        x_cible_3d = torch.minimum(haut_3d - .15*larg_3d, torch.maximum(bas_3d + .15*larg_3d, x_proj))
        eff_k = mag_k*(1.0 - is_straight)/(1.0 + .022*z_start)
        cible_v_3d = (47.5 - 1200.0*eff_k).clamp(14.0, 47.5).squeeze(1)
        bas_in_3d = (bas_3d + torch.randn_like(bas_3d)*.012).clamp(0., .92)
        haut_in_3d = torch.maximum(bas_in_3d+.06, (haut_3d + torch.randn_like(haut_3d)*.012).clamp(.08, 1.))
        bite_mask = (torch.rand(B_3d, 1) < .22).float()
        bite_row = F.one_hot(torch.randint(1, N - 1, (B_3d,)), N).float()*bite_mask
        bite_side = (torch.rand(B_3d, 1) < .5).float()
        bite_amp = torch.rand(B_3d, 1)*.20*larg_3d
        bas_in_3d = torch.minimum(haut_in_3d - .08, bas_in_3d + bite_row*bite_side*bite_amp)
        haut_in_3d = torch.maximum(bas_in_3d + .08, haut_in_3d - bite_row*(1.0 - bite_side)*bite_amp)
        rs = torch.rand(B_std, 6)
        vir = (rs[:,0:1] - .5)*.76
        dec = (rs[:,1:2] - .5)*.28
        centre_s = .5 + dec*(1.0 - .4*t_lin) + vir*(t_lin**2)
        ray_s = (.26 + .16*rs[:,2:3])*(1.0 - .65*t_lin)
        bas_s = (centre_s - ray_s).clamp(0., .92)
        haut_s = torch.maximum(bas_s+.08, (centre_s + ray_s).clamp(.08, 1.))
        larg_s = (haut_s - bas_s).clamp(min=.04)
        x_cible_s = bas_s + larg_s*(.5 + torch.tanh(vir*5.0)*profil_s).clamp(.20, .80)
        cible_v_s = (46.0 - 55.0*vir.abs().squeeze(1)).clamp(15.0, 46.0)
        bas_in_s = (bas_s + torch.randn_like(bas_s)*.008).clamp(0., .92)
        haut_in_s = torch.maximum(bas_in_s+.06, (haut_s + torch.randn_like(haut_s)*.008).clamp(.08, 1.))
        bas_clean = torch.cat((bas_3d, bas_s), dim=0)
        haut_clean = torch.cat((haut_3d, haut_s), dim=0)
        bas_in = torch.cat((bas_in_3d, bas_in_s), dim=0)
        haut_in = torch.cat((haut_in_3d, haut_in_s), dim=0)
        x_cible = torch.cat((x_cible_3d, x_cible_s), dim=0)
        re = torch.rand(B, 2)
        etat = torch.stack((re[:,0], re[:,1]*2.0 - 1.0), dim=1)
        cible_v = (torch.cat((cible_v_3d, cible_v_s), dim=0) - 3.0*etat[:,1].abs()).clamp(12.0, 48.0)
        x,v = modele(torch.stack((bas_in,haut_in),dim=-1),etat)
        delta = x[:,1:] - x[:,:-1]
        courbure = delta[:,1:]-delta[:,:-1]
        larg = (haut_clean-bas_clean).clamp(min=.04)
        s_pos = (x-bas_clean)/larg
        marge_pen = F.relu(.14-s_pos).square().mean() + F.relu(s_pos-.86).square().mean()
        cout = (4.5*(x-x_cible).square().mean() + 26.0*courbure.square().mean()
                + 8.0*marge_pen + ((v-cible_v)/38.0).square().mean())
        optim.zero_grad()
        cout.backward()
        optim.step()
    return modele.eval()

def dessiner(image, plan=None, raison=""):
    image = cv2.resize(image,(640,480))
    if image.ndim==2:
        image = cv2.cvtColor(image,cv2.COLOR_GRAY2BGR)
    if plan is not None:
        points = np.rint(plan.xy*[639,479]).astype(np.int32)
        cv2.polylines(image,[points],False,(0,240,130),3,cv2.LINE_AA)
        texte = f"Cible indicative  {plan.cible_kmh:.0f} km/h"
    else:
        texte = "Guidage indisponible"
    cv2.putText(image,texte,(12,32),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,255,255),2)
    return image

In [ ]:
def oracle_couloir(virage):
    y,x = np.mgrid[:96,:128]
    t = np.clip((.87-y/95)/.30,0,1)
    centre = .5 + virage*t*t
    largeur = .30-.18*t
    return (abs(x/127-centre)<largeur).astype(np.float32)

def tester_a2(modele):
    plans = [modele.proposer(oracle_couloir(k),8.,0.) for k in (-.23,0.,.23)]
    assert plans[0].xy[-1,0]<.4 and plans[2].xy[-1,0]>.6
    assert max(plans[0].cible_kmh,plans[2].cible_kmh)<plans[1].cible_kmh
    assert np.max(abs(plans[1].xy[:,0]-.5))<.09
    bloque = oracle_couloir(0)
    bloque[66:68,:] = 0  # aussi entre deux lignes de mesure
    for invalide in (np.zeros_like(bloque),np.ones_like(bloque),bloque,np.full_like(bloque,np.nan)):
        try:
            modele.proposer(invalide,8.,0.)
        except GuidageIndisponible:
            continue
        raise AssertionError("Un plan a traversé une zone interdite")
    # Une bande de piste devant le pare-brise doit être utilisable seulement
    # dans le cadrage cockpit. Une interruption au milieu annule la ligne.
    bande = np.zeros((240,320),np.float32)
    bande[104:122,118:245] = 1
    ancien = modele.cadrage
    try:
        modele.cadrage = COCKPIT_ABARTH
        plan = modele.proposer(bande,8.,0.)
        assert np.array_equal(np.rint(plan.xy[:,1]*239),np.rint(LIGNES_COCKPIT*239))
        bande[112,:] = 0
        try:
            modele.proposer(bande,8.,0.)
        except GuidageIndisponible:
            pass
        else:
            raise AssertionError("Une coupure du passage cockpit doit annuler le plan")
        bande_capot = np.zeros((240,320),np.float32)
        bande_capot[122:180,60:260] = 1
        modele.cadrage = CAPOT_ABARTH
        plan = modele.proposer(bande_capot,8.,0.)
        ys_capot = np.rint(plan.xy[:,1]*239)
        assert ys_capot[0]==176 and ys_capot[-1]>=122 and np.all(np.diff(ys_capot)<0)
        bande_capot[151,:] = 0
        try:
            modele.proposer(bande_capot,8.,0.)
        except GuidageIndisponible:
            pass
        else:
            raise AssertionError("Une coupure du passage capot doit annuler le plan")
    finally:
        modele.cadrage = ancien
    return {'directions':['gauche','droit','droite'],'cibles_kmh':[p.cible_kmh for p in plans],
            'refus_attendus':4}

In [ ]:
def tester_chaine_humaine(a1,a2):
    rng = np.random.default_rng(564)
    reussis = 0
    franchissements = 0
    for _ in range(40):
        image,masque = scene_synthetique(rng)
        _,p = a1.observer(image)
        try:
            plan = a2.proposer(p,6.,0.)
        except GuidageIndisponible:
            continue
        visuel = dessiner(image,plan)
        assert visuel.shape==(480,640,3)
        h,w = masque.shape
        trace = np.zeros_like(masque)
        cv2.polylines(trace,[np.rint(plan.xy*[w-1,h-1]).astype(np.int32)],False,1,2)
        franchissements += int(np.any((trace>0)&(masque!=1)))
        reussis += 1
    assert reussis>=16, "Couverture insuffisante des scènes synthétiques"
    assert franchissements==0, f"{franchissements} lignes hors du masque de référence"
    return dict(scenes=40,trajectoires_affichees=reussis,franchissements=franchissements)